# MODE 2 — M2_F06 AIRCRAFT CARRIER — PRODUCTION

> Phase 8 — Dual Pipeline Doctrine — v1.0.0

```
INPUT  : IN_FINAL_FRAMES/*.png + IN_AUDIO/*.wav (optionnel)
OUTPUT : OUT_FINAL_MOVIE/FINAL_M2.mp4
```

**LOI R-04 :** Choix overlay BINAIRE — OUI (audio+texte) ou NON (vidéo brute).

In [ ]:
# ── CELLULE 0 — CONFIGURATION OPÉRATEUR ──────────────────────

# LOI R-04 : overlay binaire
OVERLAY       = "no"    # "yes" ou "no"
OVERLAY_TEXT  = ""      # Texte à graver (si OVERLAY=yes)
AUDIO_FILE    = None    # None = auto IN_AUDIO/*

# RIFE
TARGET_FPS    = 60      # 60 ou 120
NO_RIFE       = False   # True = skip RIFE

# Upscale
NO_UPSCALE    = True    # True = skip Real-CUGAN (plus rapide)
UPSCALE_FACTOR = 2

# Encode
FORMAT        = "h265"  # h265 | av1 | prores
SOURCE_FPS    = 24

DRY_RUN       = False
VERBOSE       = True

In [ ]:
# ── CELLULE 1 — LANCEMENT ─────────────────────────────────────
import subprocess, sys
from pathlib import Path

script = Path("EXO_M2_F06_CARRIER.py")
cmd = [sys.executable, str(script), "--overlay", OVERLAY]

if OVERLAY_TEXT:  cmd += ["--text", OVERLAY_TEXT]
if AUDIO_FILE:    cmd += ["--audio", AUDIO_FILE]
cmd += ["--target-fps", str(TARGET_FPS)]
cmd += ["--source-fps", str(SOURCE_FPS)]
cmd += ["--format", FORMAT]
cmd += ["--upscale-factor", str(UPSCALE_FACTOR)]
if NO_RIFE:       cmd.append("--no-rife")
if NO_UPSCALE:    cmd.append("--no-upscale")
if DRY_RUN:       cmd.append("--dry-run")
if VERBOSE:       cmd.append("--verbose")

print(f"Commande : {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False, text=True)
print(f"\nCode retour : {result.returncode}")

In [ ]:
# ── CELLULE 2 — RAPPORT ───────────────────────────────────────
import json
from pathlib import Path

report_path = Path("../OUT_REPORT/m2_f06_report.json")
if not report_path.exists():
    print("Rapport introuvable")
else:
    with open(report_path) as f:
        r = json.load(f)
    icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"{icon} STATUS : {r['status']}")
    print(f"   Overlay : {r.get('overlay_mode')}")
    print("\n   Étapes :")
    for step, data in r.get("steps", {}).items():
        ok_icon = "✅" if data.get("ok") else "❌"
        skip = " (SKIPPED)" if data.get("skipped") else ""
        print(f"     {ok_icon} {step}{skip}")
    out = r.get("outputs", {})
    if out.get("final"):
        fp = Path(out["final"])
        size = fp.stat().st_size / (1024*1024) if fp.exists() else 0
        print(f"\n   ✅ FINAL : {fp.name} ({size:.1f} MB)")
        print("\n   ══════════════════════════════")
        print("   PIPELINE MODE 2 TERMINÉ")
        print("   Livrable : OUT_FINAL_MOVIE/FINAL_M2.mp4")